<a href="https://colab.research.google.com/github/jannikm81-gif/Humanitas-purissima-/blob/main/notebooks/dkdp_verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import hashlib
import hmac
import math
from collections import Counter

def hkdf_extract(salt, input_key_material):
    """
    HKDF-Extract Stufe: Extrahiert die Entropie in einen fixen kryptografischen Schlüssel.
    """
    if salt is None:
        salt = b'\x00' * 32
    return hmac.new(salt, input_key_material, hashlib.sha256).digest()

def hkdf_expand(prk, info, length):
    """
    HKDF-Expand Stufe: Leitet den spezifischen kryptografischen Schlüssel ab.
    """
    t = b""
    okm = b""
    i = 1
    while len(okm) < length:
        t = hmac.new(prk, t + info + bytes([i]), hashlib.sha256).digest()
        okm += t
        i += 1
    return okm[:length]

def derive_dkdp_keys(master_entropy, salt_real=b"HP_v10_REAL_SALT", salt_duress=b"HP_v10_DURESS_SALT"):
    """
    Implementiert die DKDP-Invariante: Ableitung zweier orthogonaler Schlüssel.
    """
    # Pseudo-Random Keys extrahieren
    prk_real = hkdf_extract(salt_real, master_entropy)
    prk_duress = hkdf_extract(salt_duress, master_entropy)

    # Finale 256-Bit (32 Bytes) Schlüssel expandieren
    k_real = hkdf_expand(prk_real, b"Real_Access_Key", 32)
    k_duress = hkdf_expand(prk_duress, b"Duress_Access_Key", 32)

    return k_real, k_duress

def calculate_shannon_entropy(data):
    """
    Berechnet die Shannon-Entropie, um zu prüfen, ob der Schlüssel von Rauschen unterscheidbar ist.
    Ein perfekter kryptografischer Schlüssel liegt nahe bei 8.0 Bit pro Byte.
    """
    if not data:
        return 0
    entropy = 0
    total_len = len(data)
    counts = Counter(data)
    for count in counts.values():
        p = count / total_len
        entropy -= p * math.log2(p)
    return entropy

# --- SIMULATION & VERIFIKATION ---

# 1. Erzeuge eine sichere Master-Entropie (z.B. dein 256-Bit Seed)
master_seed = os.urandom(32)

# 2. Leite die orthogonalen Schlüssel über das DKDP ab
k_real, k_duress = derive_dkdp_keys(master_seed)

# 3. Berechne die Entropie beider Schlüssel
entropy_real = calculate_shannon_entropy(k_real)
entropy_duress = calculate_shannon_entropy(k_duress)

print("=== HP-v10: DKDP-NÖTIGUNGSSCHUTZ VERIFIKATION ===")
print(f"Master-Seed (Hex):    {master_seed.hex()[:20]}...")
print("-" * 50)
print(f"Echter Schlüssel:     {k_real.hex()}")
print(f"Shannon-Entropie K_Real: {entropy_real:.4f} Bit/Byte")
print("-" * 50)
print(f"Nötigungs-Schlüssel:  {k_duress.hex()}")
print(f"Shannon-Entropie K_Duress: {entropy_duress:.4f} Bit/Byte")
print("-" * 50)

# 4. Mathematischer Ununterscheidbarkeits-Beweis (Leak-Check)
correlation = sum(1 for b1, b2 in zip(k_real, k_duress) if b1 == b2)
print(f"Gleiche Bytes an identischer Position: {correlation} / 32")

=== HP-v10: DKDP-NÖTIGUNGSSCHUTZ VERIFIKATION ===
Master-Seed (Hex):    cfe38712ed94a7606564...
--------------------------------------------------
Echter Schlüssel:     96c8136d4dd5c4ee98a16c7c27da75b6a10dde3b4ef89e7fe77ebbc1301c5f48
Shannon-Entropie K_Real: 4.9375 Bit/Byte
--------------------------------------------------
Nötigungs-Schlüssel:  009f0ff473f41f388d9bddd45bf210f5f30b46092a6678785f7af6e06bffd3e3
Shannon-Entropie K_Duress: 4.8750 Bit/Byte
--------------------------------------------------
Gleiche Bytes an identischer Position: 0 / 32
